# bring Trafilatura for extracting from web page 

(not using funcky monkey)

get the article

In [4]:
import trafilatura
import re 
import unicodedata
from sentence_transformers import SentenceTransformer 
import numpy as np 
import json
import httpx






In [5]:

def clean_up(text):
    text = unicodedata.normalize("NFC", text)          # 1) normalize Unicode
    text = re.sub(r"[ \t]+", " ", text)                # 2) collapse runs of spaces/tabs → single space
    text = re.sub(r"\n{3,}", "\n\n", text)             # 3) collapse 3+ newlines → exactly two
    text = text.strip()                                 # 4) trim leading/trailing whitespace
    return text

def fetch_article(url):
    article = trafilatura.fetch_url(url)
    if not article:
        raise ValueError("url failed")
    metadata = trafilatura.extract_metadata(article)
    if metadata and metadata.title:
        article_title = metadata.title
    extract_json = trafilatura.extract(article, output_format='json')
    if not extract_json:
        raise ValueError("json failed")
    
    data = json.loads(extract_json)
    text = (data.get("text") or "").strip()
    text = clean_up(text)
    
    if not text:
        raise ValueError("no text")
    
    return article_title, text 

tare it to chunks and store

In [6]:


def clean_and_segment(text):
    text = re.sub(r'\s+', ' ', text).strip()
    
    chunks = re.split(r'(?<=[.!?]) +', text)
    grouped = []
    temp = ""
    for sentence in chunks:
        temp += sentence + " "
        if len(temp) > 80:
            grouped.append(temp.strip())
            temp = " "
    if temp:
        grouped.append(temp.strip())
        
    return grouped

bring model that can recognize meaning 

In [ ]:

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def embed_text(text: str) -> np.ndarray:

    if text is None or text.strip() == "":
        return None
    segments = [seg.strip() for seg in text.split("\n") if seg.strip()]
    
    embeddings = model.encode(segments)
    embeddings = np.array(embeddings)
    
    if embeddings.ndim == 1:
        return embeddings
    
    avg_embedding = np.mean(embeddings, axis=0)
    return avg_embedding

now save the vectore from the  model for future usege

In [8]:


def save_article(title, chunks,  embeddings ,path= '\articles'):
    np.save(f"{path}{title}_embeddings.npy", embeddings)
    with open(f"{path}{title}_text.json", 'w', encoding= 'utf8') as f:
        json.dump(chunks, f, ensure_ascii= False, indent= 2)

# crawl the web


In [9]:
# import urllib.robotparser as robotparser

# USER_AGENT = "ArticleCompareBot/0.1 (+contact@example.com)"
# DOMAIN_LIMITERS = {}  # domain -> AsyncLimiter

# def limiter_for(domain, rps=1):
#     if domain not in DOMAIN_LIMITERS:
#         DOMAIN_LIMITERS[domain] = AsyncLimiter(rps, 1)
#     return DOMAIN_LIMITERS[domain]


# def canonicalize(url):
#     u = url.strip()
#     u = u.split("#")[0]
#     if u.endswith("/"): u = u[:-1]
#     return u

# def domain_of(url): return urlparse(url).netloc

In [10]:
%run learning_to_scrape.ipynb



In [11]:
html = fetch_html("https://www.ynet.co.il/home/0,7340,L-8,00.html")
links = extract_links("https://www.ynet.co.il/", html)

TypeError: urljoin() missing 1 required positional argument: 'url'

In [ ]:
print(links)

{'ynet - מבזקים': 'https://www.ynet.co.il/news/category/184', 'ynet - דואר אדום': 'https://www.ynet.co.il/redmail', 'ynet - חדשות, כלכלה, ספורט ובריאות - דיווחים שוטפים מהארץ ומהעולם': 'https://www.ynet.co.il/home/0,7340,L-8,00.html', 'ynet - פלוס': 'https://www.ynet.co.il/plus', 'ynet - חדשות -  סיקורים, מבזקים וכתבות מהארץ והעולם': 'https://www.ynet.co.il/news', 'ynet - radio': 'https://www.ynet.co.il/radio', 'ynet - כלכלה - ערוץ פיננסים הכולל פרשנויות, חדשות וכתבות': 'https://www.ynet.co.il/economy', 'ynet - חדשות ספורט -  כל מה שקורה בכדורגל, הכדורסל ושאר הענפים': 'https://www.ynet.co.il/sport', 'ynet - תרבות  - כל מה שמעניין בטלוויזיה, הקולנוע הספרות והבידור': 'https://www.ynet.co.il/entertainment', 'pplus - פנאי פלוס': 'https://pplus.ynet.co.il/homepage', 'ynet - בריאות - חדשות, מחקרים והתפתחויות בתחומי הרפואה': 'https://www.ynet.co.il/health', 'ynet - רכב - סיקורים, חדשות, מבחני דרכים וכל מה שמרתק בתחום': 'https://www.ynet.co.il/wheels', 'ynet - דיגיטל - כל מה שחדש בתחום המחשוב,

In [ ]:
title, articel = fetch_article("https://www.ynet.co.il/news/article/bk11kbatkze#autoplay")
grouped = clean_and_segment(articel)
vector_of_art =  model.encode(grouped, show_progress_bar=True,convert_to_numpy=True, normalize_embeddings= True)
article_emb = vector_of_art.mean(axis=0)
article_emb /= (np.linalg.norm(article_emb) + 1e-10)
titles = [title for title in links.keys()]
vector_of_title = model.encode(titles, show_progress_bar=True, convert_to_numpy= True, normalize_embeddings=True )
sims = vector_of_title @ article_emb
order = np.argsort(-sims) 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
TOP_K = 10
urls =  [url for url in links.values()]
THRESHOLD = 0.6
for i in order[:TOP_K]:
    if sims[i] >= THRESHOLD:
        print(f"{sims[i]:.3f} | {titles[i]} -> {urls[i]}")

0.688 | חמאס: נשחרר את החלל החטוף הדר גולדים בשעה 14:00 -> https://www.ynet.co.il/news/article/bk11kbatkze#autoplay
0.638 | שורדות זנות בישראל: "יש יובל מצחיקה, אבל מה שעשתה בלילה - סוד גדול" -> https://www.ynet.co.il/news/article/skwesercxl
0.633 | גם בזכות הישראלים: הפער שהביא להפועל ירושלים את גביע ווינר | טור -> https://www.ynet.co.il/sport/article/bjhluetjze
